# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the <b>FAIR<sup>2</sup> dataset</b> using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and prepare for further exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and view top-level metadata
meta = dataset.metadata
print(f"Name: {meta.name}")
print(f"Description: {meta.description}")
print(f"Version: {getattr(meta, 'version', 'N/A')}")
print(f"License: {getattr(meta, 'license', 'N/A')}")

## 2. Data Overview
Review the available record sets and their field `@id` values using Croissant metadata.

In [ ]:
# List all record sets and their fields, using their `@id`
if hasattr(meta, 'record_sets'):
    record_sets = meta.record_sets
elif hasattr(meta, 'recordSet'):
    record_sets = meta.recordSet
else:
    record_sets = []
    print('No record sets found in metadata.')

record_set_ids = []

for rec in record_sets:
    print(f"Record Set name: {getattr(rec, 'name', '--')}  |  @id: {rec.id}")
    record_set_ids.append(rec.id)
    fields = getattr(rec, 'fields', getattr(rec, 'field', []))
    if not fields:
        continue
    for field in fields:
        print(f"    Field: {getattr(field, 'name', '--')}  @id: {field.id if hasattr(field, 'id') else field['@id']}")
if not record_set_ids:
    print('No record sets (`recordSet`) explicitly declared. The dataset may have a single main table or use indirect file links.')

## 3. Data Extraction
Load data from all available record sets (or main record set/file objects) into Pandas DataFrames. Use `@id` references as required.

In [ ]:
# If no recordSet, try to discover data file objects
if not record_set_ids and hasattr(meta, 'distribution'):
    # Find a distribution with tabular content (e.g. CSV)
    for dist in meta.distribution:
        url = getattr(dist, 'contentUrl', None) or getattr(dist, 'contentURL', None)
        enc_format = getattr(dist, 'encodingFormat', None)
        print(f"Distribution @id: {dist.id if hasattr(dist, 'id') else dist['@id']}")
        print(f"   URL: {url}")
        print(f"   Format: {enc_format}")
        # If CSV, load as DataFrame
        if url and enc_format and 'csv' in enc_format.lower():
            df = pd.read_csv(url)
            dataframes = {dist.id if hasattr(dist, 'id') else dist['@id']: df}
            target_id = dist.id if hasattr(dist, 'id') else dist['@id']
            print(df.head())
            break
    else:
        dataframes = {}
        target_id = None
else:
    # Use croissant's records() generator
    dataframes = {}
    target_id = None
    for rec_id in record_set_ids:
        records = list(dataset.records(record_set=rec_id))
        if records:
            dataframes[rec_id] = pd.DataFrame(records)
            print(f"Loaded DataFrame for record set @id: {rec_id}  |  Shape: {dataframes[rec_id].shape}")
            print(f"Columns: {dataframes[rec_id].columns.tolist()}")
            target_id = rec_id
        else:
            print(f"No records found for record set {rec_id}.")if not dataframes:
    print('No tabular data could be loaded.')

## 4. Exploratory Data Analysis (EDA)
Apply some filtering, normalization, and grouping operations – all referencing the field columns by their `@id`. This section demonstrates how to analyze numeric or categorical fields, remove outliers, and create normalized metrics.

In [ ]:
import numpy as np

# Select the DataFrame for further EDA
if dataframes:
    if target_id is None:
        target_id = list(dataframes.keys())[0]
    df = dataframes[target_id]
    print(f"Working with DataFrame from record/distribution: {target_id}")
    # List columns and determine a numeric field
    print('Columns:', df.columns.tolist())
    # Attempt to auto-detect a numeric column
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
    else:
        # Try to coerce to numeric
        col_found = False
        for col in df.columns:
            coerced = pd.to_numeric(df[col], errors='coerce')
            if coerced.notna().sum() > 0 and coerced.notna().mean() > 0.5:
                df[col] = coerced
                numeric_field = col
                col_found = True
                break
        if not col_found:
            numeric_field = df.columns[0]
    print(f"Chosen numeric field for demo: {numeric_field}")

    # Filter: e.g., field > threshold
    # Choose threshold as 1 std above mean, or 10 if values are large
    mean_val = df[numeric_field].mean()
    std_val = df[numeric_field].std()
    threshold = mean_val + std_val if mean_val > 10 else 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records where {numeric_field} > {threshold:.2f}. {filtered_df.shape[0]} records remain.")

    # Normalization
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, norm_col]].head())

    # Try to group by a categorical field if possible
    group_field = None
    for col in df.columns:
        if df[col].dtype == object and df[col].nunique() > 1 and df[col].nunique() < filtered_df.shape[0] / 2:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped mean of '{numeric_field}' by '{group_field}':")
        print(grouped_df.head())
else:
    print('No DataFrame available for EDA.')

## 5. Visualization
Visualize numeric field distributions and their relationships to available categorical/grouping fields, referencing columns by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and target_id in dataframes:
    df = dataframes[target_id]
    # Histogram of the main numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot by group_field (if found)
    if group_field:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=90)
        plt.show()
else:
    print('No DataFrame or numeric field for visualization.')

## 6. Conclusion
In this notebook, we:
- Loaded dataset metadata via the Croissant schema using `mlcroissant`.
- Discovered available record sets and their fields, all via `@id` referencing.
- Loaded tabular data and explored numeric summaries and groupings.
- Performed basic EDA and visualizations, referencing fields by their `@id`.
  
This process can be adapted for downstream analysis, machine learning workflows, or additional data integration. Refer to the [FAIR<sup>2</sup> Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) for full field and structure details.